In [2]:
%pip install -U langchain

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install langchain-community langchain-huggingface faiss-cpu sentence-transformers transformers pandas datasets

Note: you may need to restart the kernel to use updated packages.


In [4]:
import langchain
import pandas
import faiss
import sentence_transformers
import transformers
import datasets

print("All packages installed successfully")

All packages installed successfully


In [5]:
from datasets import load_dataset
dataset = load_dataset("batuhanmtl/job-skill-set")

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['job_id', 'category', 'job_title', 'job_description', 'job_skill_set'],
        num_rows: 1167
    })
})

In [7]:
df = dataset["train"].to_pandas()

In [8]:
df.sample(5)

,job_id,category,job_title,job_description,job_skill_set
93,3887474527,HR,Human Resources Generalist,WE'RE NOT JUST IN THE PACKAGING BUSINESS–WE'RE...,"['human resources administration', 'recruitmen..."
214,3894925340,HR,Human Resource Manager,GLOBAL INDUSTRIALCORPORATE OVERVIEWFOR OVER 70...,"['HR Management', 'Onboarding', 'Employee Rela..."
198,3890896173,HR,Human Resources Coordinator,HUMAN RESOURCES COORDINATOR | $25-30 AN HOUR |...,"['HR administrative paperwork', 'employee onbo..."
375,3905867221,INFORMATION-TECHNOLOGY,Information Technology Manager,THE IT MANAGER WILL PRIMARILY BE RESPONSIBLE F...,"['IT management', 'budget management', 'networ..."
1000,3902326947,BUSINESS-DEVELOPMENT,Business Development Representative,POSITION PURPOSE\nTHE BUSINESS DEVELOPMENT REP...,"['sales skills', 'client presentations', 'nego..."


In [9]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():
    doc = Document(
        page_content=row["job_description"],
        metadata={
            "job_id": row["job_id"],
            "category": row["category"],
            "job_title": row["job_title"],
            "job_skill_set": row["job_skill_set"]
        }
    )
    documents.append(doc)

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size= 800, chunk_overlap = 100)
chunks = splitter.split_documents(documents)

In [11]:
len(chunks)

8122

In [12]:
print(chunks[42].page_content)

AD&D, FSA, 401(K) RETIREMENT PLAN, AND AFLAC BENEFIT OFFERINGSB. REVIEW AND ENSURE ICS BENEFITS PROGRAMS ARE COMPETITIVE, COST‐EFFECTIVE, AND SATISFY EMPLOYEE NEEDS.C. PARTNER WITH INSURANCE BROKERS AND CARRIERS TO RESOLVE BENEFIT ISSUES.D. MANAGE WEB‐BASED ENROLLMENTS DIRECTLY ON THE CARRIER PORTAL AND IN THE IN‐HOUSE PAYROLL SYSTEM TOENSURE TIMELY MEMBERSHIP ID CARDS ARE PROCESSED, AND CONTRIBUTION DEDUCTIONS ARE SET UP IN PAYROLL.E. MANAGE OPEN ENROLLMENT AND THE DAILY ADMINISTRATION OF COMPANY BENEFITS, AND RETIREMENT PLANS AND UTILIZE POSITIVE PROFESSIONAL RELATIONSHIPS WITH BROKERS, CARRIERS, THIRD‐PARTY ADMINISTRATORS, AND COLLEAGUES TO MAXIMIZE A POSITIVE EMPLOYEE EXPERIENCE.F. LEAD COMPANY‐WIDE BENEFIT PLAN/PROGRAM RESEARCH, ANALYSIS, AND PROJECTS. COLLABORATE WITH THELEADERSHIP


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

In [14]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
vector =  embeddings.embed_query(chunks[0].page_content)


In [16]:
print(len(vector))

384


In [17]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(chunks, embeddings)

C:\Users\rites\AppData\Local\Temp\ipykernel_12264\3484710893.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [18]:
vector_store.index_to_docstore_id

{0: 'daa1fa6c-72d2-4dc3-97e6-a119ea20a5b9',
 1: '9222f9b6-020f-4789-b262-ca0e63a500f9',
 2: '94b8947e-d45f-404f-a8e5-230a7aae1177',
 3: 'af011bc5-6c44-41b6-be23-6f2df5210b5a',
 4: '837c2665-d0eb-4f5b-b347-ed9db35fa71e',
 5: 'b59df728-de3e-4f05-ae26-1039385b852e',
 6: '77f8f7b0-3231-4afb-80d1-96ff35902adb',
 7: 'a457a470-9984-46d9-97d2-8a986dedb93a',
 8: 'c1e41529-2318-4fc9-985b-28d18ac6b581',
 9: 'e6114215-7108-4cc6-a7f8-8a84ab745ec5',
 10: '599be61a-c58d-4d13-8889-ac739f9a146c',
 11: '96f75d8e-2252-4ca0-a677-62cdea2f5812',
 12: '05b5d99d-fe9d-40a8-9888-ee7747b5d1a9',
 13: '0529504a-3f15-4c21-a0c9-58f775591187',
 14: '11c6c00f-332a-43df-9dea-44e43a0fbdb3',
 15: 'de95ceae-f059-4447-8513-b4a63c162f8a',
 16: '40a31d7a-f323-4a93-ad06-6fe3466178b9',
 17: '9b0ea62e-9da4-4edb-9ecf-fc4235828f03',
 18: 'ce968dd0-5ba6-4cbd-9dc8-dd289ead9870',
 19: '6fc49ad8-7053-4642-88a8-1e29dda297d7',
 20: '7b5a0c17-0ef4-478c-bbe0-9a569cbba43d',
 21: '40ec310d-9003-4778-956f-fee058f272ec',
 22: '814eb388-a31e-

In [40]:
vector_store.get_by_ids(['240bb5af-f601-4bad-bd36-9af54bbd06c5'])

[Document(id='240bb5af-f601-4bad-bd36-9af54bbd06c5', metadata={'job_id': 3884912913, 'category': 'HR', 'job_title': 'Human Resources Business Partner', 'job_skill_set': "['Human Resources', 'Employee Relations', 'HRIS', 'Microsoft Office (Excel, PowerPoint, Word)', 'Analytical Skills', 'Talent Acquisition', 'Compliance with Labor Laws', 'Communication', 'Coaching', 'Organizational Skills', 'Multi-tasking', 'Problem Solving', 'Teamwork', 'Adaptability', 'Project Management', 'Investigative Skills', 'Commitment to Safety']"}, page_content='PROMOTES A CULTURE OF SAFETY AND EXHIBITS SUCH BEHAVIORS.PROVIDES GENERAL COACHING TO FRONTLINE SUPERVISORS ON EMPLOYEE RELATIONS MATTERSCONDUCTS INVESTIGATIONS FOR SELECT EMPLOYEE RELATIONS CLAIMS. ACTIVELY ENGAGES IN COMMUNICATIONS WITH OTHER HR TEAM MEMBERS AROUND BEST PRACTICES AND IDENTIFIES OPPORTUNITIES TO CREATE CONSISTENCY WHERE NECESSARY SUPPORTS THE HR MANAGER WITH EMPLOYEE RELATIONS ISSUES. ENSURES FIELD OPERATIONS ARE IN COMPLIANCE WITH VA

In [43]:
# RAG pipeline starts

retriever = vector_store.as_retriever(search_type= 'similarity' , search_kwargs={"k":4})

In [21]:
results = retriever.invoke("job skill set for finance")
print(results)

[Document(id='afbabc01-cb52-49a2-b8b7-3b136c911bdd', metadata={'job_id': 3905350188, 'category': 'FINANCE', 'job_title': 'Strategic Finance Manager', 'job_skill_set': "['financial analysis', 'budget management', 'financial forecasting', 'data analysis', 'financial software', 'GAAP', 'communication', 'analytical skills', 'interpersonal skills', 'collaboration', 'problem solving', 'innovation', 'financial planning', 'financial modeling', 'advanced Excel', 'eCommerce', 'marketplace experience']"}, page_content='QUALIFICATIONS:BACHELOR’S DEGREE IN FINANCE, ACCOUNTING, ECONOMICS, OR RELATED FIELD.HAS 6+ YEARS OF EXPERIENCE IN STRATEGIC FINANCE, FP&A, BIG 4 ACCOUNTING, INVESTMENT BANKING, OR CORPORATE FINANCE.PRIOR EXPERIENCE IN HIGH-GROWTH OR STARTUP ENVIRONMENTS IS REQUIRED.ADVANCED FINANCIAL MODELING TECHNIQUES AND ABILITIES WITH A DEEP UNDERSTANDING OF BEST FINANCIAL MODELING PRACTICES.EXCEPTIONAL ANALYTICAL SKILLS AND THE ABILITY TO SYNTHESIZE COMPLEX FINANCIAL INFORMATION INTO CLEAR, C

In [22]:
from langchain_core.prompts import ChatPromptTemplate
chat_template = ChatPromptTemplate([
  ('system' , 'You are a helpful job information expert.'),
   ('human' , 'Context:\n {context}\n\n Question:\n{question}')

])

In [23]:
context = "\n\n".join(
    doc.page_content for doc in results
)

In [24]:
question = "job skill set for information technology"
results = retriever.invoke(question)

In [25]:
prompt = chat_template.invoke({'question':question , 'context':context})

In [26]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a helpful job information expert.', additional_kwargs={}, response_metadata={}), HumanMessage(content="Context:\n QUALIFICATIONS:BACHELOR’S DEGREE IN FINANCE, ACCOUNTING, ECONOMICS, OR RELATED FIELD.HAS 6+ YEARS OF EXPERIENCE IN STRATEGIC FINANCE, FP&A, BIG 4 ACCOUNTING, INVESTMENT BANKING, OR CORPORATE FINANCE.PRIOR EXPERIENCE IN HIGH-GROWTH OR STARTUP ENVIRONMENTS IS REQUIRED.ADVANCED FINANCIAL MODELING TECHNIQUES AND ABILITIES WITH A DEEP UNDERSTANDING OF BEST FINANCIAL MODELING PRACTICES.EXCEPTIONAL ANALYTICAL SKILLS AND THE ABILITY TO SYNTHESIZE COMPLEX FINANCIAL INFORMATION INTO CLEAR, CONCISE REPORTS.EXPERTISE IN DATA ANALYSIS WITH ADVANCED EXCEL/GOOGLE SHEET SKILLS AND EXPERIENCE WITH FINANCIAL SOFTWARE AND SYSTEMS.EXCEPTIONAL ANALYTICAL SKILLS AND THE ABILITY TO SYNTHESIZE COMPLEX FINANCIAL INFORMATION INTO CLEAR, CONCISE REPORTS.STRONG INTERPERSONAL AND COMMUNICATION SKILLS, WITH THE\n\nBUILD STRONG INDUSTRY KNOWLEDGE O

In [27]:
%pip install -U langchain-huggingface huggingface_hub

  Using cached huggingface_hub-2.2.0-py3-none-any.whl.metadata (16 kB)
Note: you may need to restart the kernel to use updated packages.


In [28]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [29]:
%pip install -U langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [30]:
from google import genai

client = genai.Client()


In [31]:
from langchain_google_genai import ChatGoogleGenerativeAI
model= ChatGoogleGenerativeAI(model = 'gemini-3.5-flash-lite')

In [32]:
response = model.invoke(prompt)

print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'Based on the provided context, the primary job skill sets mentioned across the various roles (which heavily lean toward finance, accounting, and investment analysis) that overlap with or involve **Information Technology (IT), data, and systems** include:\n\n* **Technology & Software Experience:** Experience in technology and/or SaaS (Software-as-a-Service) environments.\n* **Financial Software & Systems:** Expertise and experience working with specialized financial software and systems, as well as financial analysis software.\n* **Data Analysis:** Strong expertise in data analysis, including interpreting data to analyze technology trends (as well as market and economic trends).\n* **Advanced Spreadsheet Skills:** Advanced proficiency in Microsoft Excel and Google Sheets (used for data processing, modeling, and technical reporting).\n* **Technical Modeling:** Advanced and expert-level financial modeling techniques and capabilities.', 'extras': {'signature': 'E

In [33]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

In [34]:
from langchain_core.runnables import RunnablePassthrough


In [35]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | chat_template
    | model
)

In [36]:
response = rag_chain.invoke(
    "job skill set for information technology"
)


In [37]:
print(response.content)

[{'type': 'text', 'text': 'Based on the provided context, the required and preferred **Information Technology (IT) skill set** spans several roles (IT Support, Network/System Administration, and IT Education). Here is the comprehensive breakdown of the IT skill set mentioned in the text:\n\n### **1. Hardware, Software & Technical Support**\n* **PC & Enterprise Hardware:** Ability to perform PC-based hardware duties, troubleshooting, and experience with printers, copiers, phone & voicemail systems, and media equipment.\n* **Operating Systems & Software:** Strong knowledge of Microsoft environments, including Windows Operating Systems, MS Office Applications (Word, PowerPoint, Excel, Outlook), and server backup software.\n* **Enterprise & Support Experience:** Minimum of 2 to 3 years of experience in IT desktop support roles (preferably in an enterprise-level environment).\n\n### **2. Networking & Infrastructure**\n* **Networking Concepts:** Familiarity with networking concepts, includin

In [38]:
%pip install fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.


In [39]:
vector_store.save_local("faiss_index")

print("FAISS index saved successfully!")

FAISS index saved successfully!
